# Pipeline Thu thập, Xử lý và Chuẩn bị Dữ liệu (Wikipedia -> N-gram)

Quá trình này bao gồm 4 phần chính:
1. **Mount Google Drive & Cài đặt**: Khởi tạo môi trường và các thư viện cần thiết.
2. **Crawl Wikipedia tiếng Việt**: Gọi MediaWiki API để tải toàn bộ bài viết mới nhất (tích hợp cơ chế checkpoint chống gián đoạn).
3. **Làm sạch & Deduplicate**: Loại bỏ HTML, bảng biểu, template và áp dụng hash SHA-256 để lọc các bài/đoạn văn trùng lặp.
4. **Tokenization & Gán nhãn**: Tách từ tiếng Việt và áp dụng Sliding Window để tạo tập dữ liệu (X, y) cho mô hình N-gram.

## Phần 0: Cài đặt thư viện & Import

In [4]:
!pip install loguru datasets requests pyarrow tqdm underthesea pandas

import hashlib
import html as _html
import json
import os
import re
import time
import unicodedata
import urllib.parse
from pathlib import Path
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import requests
from datasets import Dataset, load_dataset
from loguru import logger
from tqdm.auto import tqdm
from underthesea import word_tokenize, sent_tokenize
import logging

### Mount Google Drive

Mount Drive để lưu trữ dữ liệu lâu dài giữa các session Colab (tránh mất data khi runtime reset).

In [5]:
from google.colab import drive
drive.mount('/content/drive')
path = '/content/drive/MyDrive/DM'
os.chdir(path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Phần 1: Crawl toàn bộ Wikipedia tiếng Việt

In [6]:
# --- Cấu hình Wikipedia Crawler ---
API_ENDPOINT = "https://vi.wikipedia.org/w/api.php"
USER_AGENT = (
    "NLP-NextWordPredictor/1.0 "
    "(nguyentranminhnho123@gmail.com) "
    "Python/3.x requests/2.x"
)
BATCH_SIZE = 50
CONTENT_BATCH = 50

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

def make_session() -> requests.Session:
    """Tạo HTTP session với User-Agent chuẩn."""
    session = requests.Session()
    session.headers.update({"User-Agent": USER_AGENT})
    return session

def _parse_retry_after(value: str | None, default: int) -> int:
    """Parse header Retry-After thành số giây chờ."""
    if value is None: return default
    try: return int(value)
    except ValueError: return default

def api_get(session: requests.Session, params: dict, retries: int = 5) -> dict:
    """Gọi MediaWiki API GET với retry logic."""
    params.setdefault("format", "json")
    params.setdefault("formatversion", "2")
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(API_ENDPOINT, params=params, timeout=30)
            if resp.status_code == 429:
                if attempt == retries: break
                wait = _parse_retry_after(resp.headers.get("Retry-After"), default=60)
                logger.warning(f"429 Too Many Requests – waiting {wait} s (attempt {attempt}/{retries})")
                time.sleep(wait)
                continue
            resp.raise_for_status()
            try: data = resp.json()
            except ValueError as exc:
                if attempt < retries: time.sleep(2 ** attempt); continue
                raise RuntimeError("API returned invalid JSON after all retries") from exc
            if "error" in data:
                code = data["error"].get("code", "unknown")
                if code == "maxlag":
                    if attempt == retries: break
                    wait = _parse_retry_after(resp.headers.get("Retry-After"), default=5)
                    time.sleep(wait)
                    continue
                raise RuntimeError(f"API error [{code}]: {data['error'].get('info', '')}")
            return data
        except requests.RequestException as exc:
            if attempt < retries: time.sleep(2 ** attempt)
            else: raise RuntimeError(f"API request failed after {retries} attempts: {exc}") from exc
    raise RuntimeError(f"API request failed after {retries} attempts (rate-limit/maxlag)")

def load_checkpoint(path: Path) -> tuple[dict, bool]:
    """Tải checkpoint từ file JSON. Trả về (data, is_valid)."""
    if path.exists():
        try:
            with path.open("r", encoding="utf-8") as f: return json.load(f), True
        except json.JSONDecodeError: return {}, False
    return {}, True

def save_checkpoint(path: Path, data: dict) -> None:
    """Lưu checkpoint an toàn qua file tạm (.tmp) để tránh corrupt."""
    tmp_path = path.with_suffix(".tmp")
    with tmp_path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    tmp_path.replace(path)

def fetch_page_contents(session: requests.Session, page_ids: list[int], delay: float) -> dict[int, dict]:
    """Tải nội dung wikitext cho một batch các page ID."""
    params = {
        "action": "query",
        "prop": "revisions",
        "rvprop": "content",
        "rvslots": "main",
        "pageids": "|".join(map(str, page_ids)),
        "format": "json",
        "formatversion": "2"
    }
    data = api_get(session, params)
    contents = {}

    if "query" in data and "pages" in data["query"]:
        for page in data["query"]["pages"]:
            pid = page.get("pageid")
            title = page.get("title", "")
            content = ""
            if "revisions" in page and len(page["revisions"]) > 0:
                rev = page["revisions"][0]
                if "slots" in rev and "main" in rev["slots"]:
                    content = rev["slots"]["main"].get("content", "")
            contents[pid] = {"title": title, "content": content}

    time.sleep(delay)
    return contents

def crawl(output_dir: Path, max_articles: int | None, delay: float, resume: bool) -> None:
    """Crawl Wikipedia đệ quy (quét cả Thể loại con), hiển thị progress bar và hỗ trợ resume."""

    # Danh sách các thể loại gốc đã được mở rộng
    TARGET_CATEGORIES = [
        "Thể loại:Bài viết chọn lọc",
        "Thể loại:Bài viết tốt",
        "Thể loại:Lịch sử Việt Nam",
        "Thể loại:Triều đại phong kiến Việt Nam",
        "Thể loại:Trận đánh liên quan tới Việt Nam",
        "Thể loại:Chiến tranh Đông Dương",
        "Thể loại:Kháng chiến chống Pháp",
        "Thể loại:Kháng chiến chống Mỹ",
        "Thể loại:Văn học Việt Nam",
        "Thể loại:Nhà văn Việt Nam",
        "Thể loại:Tác phẩm văn học Việt Nam",
        "Thể loại:Nghệ sĩ Việt Nam",
        "Thể loại:Địa lý Việt Nam",
        "Thể loại:Tỉnh thành Việt Nam",
        "Thể loại:Thành phố trực thuộc tỉnh Việt Nam",
        "Thể loại:Sông Việt Nam",
        "Thể loại:Tiểu sử",
        "Thể loại:Nhân vật lịch sử Việt Nam",
        "Thể loại:Anh hùng dân tộc Việt Nam",
        "Thể loại:Danh nhân văn hóa Việt Nam",
        "Thể loại:Khoa học tự nhiên",
        "Thể loại:Nhân vật",
        "Thể loại:Xã hội",
        "Thể loại:Sinh học",
        "Thể loại:Lịch sử thế giới",
        "Thể loại:Địa lý học"
    ]

    output_dir.mkdir(parents=True, exist_ok=True)
    output_file = output_dir / "vi_wiki_articles.jsonl"
    checkpoint_file = output_dir / "checkpoint.json"

    if resume:
        checkpoint, checkpoint_valid = load_checkpoint(checkpoint_file)
        if not checkpoint_valid and output_file.exists():
            logger.error("Checkpoint bị hỏng nhưng file output vẫn tồn tại. Vui lòng kiểm tra lại thủ công.")
            return
    else:
        checkpoint = {}

    seen_ids = set(checkpoint.get("seen_ids", []))
    article_count = checkpoint.get("article_count", 0)
    current_cat_idx = checkpoint.get("current_cat_idx", 0)
    cm_continue = checkpoint.get("cm_continue")

    # Sử dụng mảng động để thêm các Thể loại con quét được
    dynamic_categories = checkpoint.get("dynamic_categories", TARGET_CATEGORIES.copy())
    seen_categories = set(dynamic_categories)

    logger.info(f"Bắt đầu crawl đệ quy. Output: {output_file} | Max: {max_articles or 'Không giới hạn'} | Resume: {resume}")
    session = make_session()

    open_mode = "a" if (resume and output_file.exists()) else "w"

    # Khởi tạo thanh tiến trình tqdm thay cho logger
    pbar = tqdm(total=max_articles, initial=article_count, desc="Đang cào bài viết Wiki")

    with output_file.open(open_mode, encoding="utf-8") as out_f:
        while current_cat_idx < len(dynamic_categories):
            cat_name = dynamic_categories[current_cat_idx]
            pbar.set_postfix({"Thể loại": f"{current_cat_idx + 1}/{len(dynamic_categories)}"})

            params = {
                "action": "query",
                "list": "categorymembers",
                "cmtitle": cat_name,
                "cmnamespace": "0|14", # Cào đồng thời Bài viết (0) và Thể loại con (14)
                "cmlimit": BATCH_SIZE,
                "maxlag": 5,
                "format": "json",
                "formatversion": "2"
            }

            if cm_continue:
                params["cmcontinue"] = cm_continue
                params["continue"] = "-||"

            while True:
                if max_articles and article_count >= max_articles: break

                batch_start_cmcontinue = params.get("cmcontinue")
                data = api_get(session, params.copy())

                pages_meta = data.get("query", {}).get("categorymembers", [])
                has_more = "continue" in data
                next_cm_continue = data.get("continue", {}).get("cmcontinue")

                new_page_ids = []

                # Phân loại luồng dữ liệu
                for p in pages_meta:
                    if p["ns"] == 0 and p["pageid"] not in seen_ids:
                        new_page_ids.append(p["pageid"])
                    elif p["ns"] == 14 and p["title"] not in seen_categories:
                        dynamic_categories.append(p["title"])
                        seen_categories.add(p["title"])

                reached_limit = False

                if new_page_ids:
                    for i in range(0, len(new_page_ids), CONTENT_BATCH):
                        if reached_limit: break
                        batch_ids = new_page_ids[i : i + CONTENT_BATCH]
                        contents = fetch_page_contents(session, batch_ids, delay)

                        if not contents: continue

                        for pid, article in contents.items():
                            if not article.get("content"): continue
                            record = {
                                "id": pid, "title": article["title"], "text": article["content"],
                                "url": "https://vi.wikipedia.org/wiki/" + urllib.parse.quote(article["title"].replace(" ", "_"), safe="/:")
                            }
                            out_f.write(json.dumps(record, ensure_ascii=False) + "\n")
                            seen_ids.add(pid)
                            article_count += 1

                            # Cập nhật thanh tiến trình tăng lên 1 đơn vị
                            pbar.update(1)

                            if max_articles and article_count >= max_articles:
                                reached_limit = True; break

                        out_f.flush()
                        save_checkpoint(checkpoint_file, {
                            "seen_ids": list(seen_ids),
                            "article_count": article_count,
                            "current_cat_idx": current_cat_idx,
                            "cm_continue": batch_start_cmcontinue,
                            "dynamic_categories": dynamic_categories
                        })

                if reached_limit: break

                save_checkpoint(checkpoint_file, {
                    "seen_ids": list(seen_ids),
                    "article_count": article_count,
                    "current_cat_idx": current_cat_idx,
                    "cm_continue": next_cm_continue,
                    "dynamic_categories": dynamic_categories
                })

                if not has_more: break
                params.update(data["continue"])
                time.sleep(delay)

            # Đặt lại token khi qua thể loại tiếp theo
            cm_continue = None
            if max_articles and article_count >= max_articles: break

            current_cat_idx += 1

    pbar.close() # Đóng thanh tiến trình khi hoàn thành

# --- Chạy crawler ---
WIKI_RAW_DIR = Path("/content/drive/MyDrive/DM/data/raws")

crawl(output_dir=WIKI_RAW_DIR, max_articles=50000, delay=1.0, resume=False)

Đang cào bài viết Wiki:   0%|          | 0/50000 [00:00<?, ?it/s]

## Phần 2: Làm sạch Wikitext và Loại bỏ trùng lặp (Dedup)

In [7]:
# --- Regex Patterns & Hằng số dùng cho việc làm sạch Wikitext ---
_PATTERNS = [
    (re.compile(r"", re.DOTALL), ""),
    (re.compile(r"<ref[^>]*/\s*>", re.IGNORECASE), ""),
    (re.compile(r"<ref[^>]*>.*?</ref>", re.DOTALL | re.IGNORECASE), ""),
    (re.compile(r"<(div|span|small|big|b|i|u|s|br|p|table|tr|th|td|ul|ol|li|dl|dt|dd|sub|sup|blockquote|nowiki|code|pre|gallery|imagemap|timeline|score|syntaxhighlight|source|poem|section|indicator|templatestyles|hiero|math|chem)[^>]*>", re.IGNORECASE), " "),
    (re.compile(r"</[a-z]+>", re.IGNORECASE), " "),
    (re.compile(r"<[^>]+>"), ""),
]

_MARKUP = re.compile(r"'''|''|----+")
_MAGIC = re.compile(r"__[A-Z_]+__")
_WHITESPACE = re.compile(r"[ \t]+")
_NEWLINES   = re.compile(r"\n{3,}")
_HEADERS = re.compile(r"^=+\s*(.+?)\s*=+\s*$", re.MULTILINE)
_EMPTY_HEADERS = re.compile(r"^=+\s*=+\s*$", re.MULTILINE)
_LIST_PREFIX = re.compile(r"^([*#;:]+)\s?", re.MULTILINE)
_TERMINAL_SECTION_RE = re.compile(
    r"^\s*=+\s*("
    r"Tham\s+kh\u1ea3o|Ch\u00fa\s+th\u00edch|Ghi\s+ch\u00fa|Li\u00ean\s+k\u1ebft\s+ngo\u00e0i"
    r"|Xem\s+th\u00eam|Th\u01b0\s+m\u1ee5c|\u0110\u1ecdc\s+th\u00eam|Ngu\u1ed3n\s+tham\s+kh\u1ea3o"
    r"|Ch\u00fa\s+gi\u1ea3i|Ghi\s+ch\u00fa\s+v\u00e0\s+tham\s+kh\u1ea3o"
    r"|References?|External\s+links?|See\s+also|Notes?|Bibliography|Further\s+reading"
    r")\s*=+\s*$",
    re.IGNORECASE | re.MULTILINE,
)

def remove_terminal_sections(text: str) -> str:
    """Cắt bỏ toàn bộ văn bản từ mục Tham khảo / Liên kết ngoài trở đi."""

    # Biểu thức chính quy tìm các tiêu đề kết bài phổ biến ở Wiki tiếng Việt
    terminal_regex = re.compile(
        r"^\s*=+\s*("
        r"Tham\s+khảo|Chú\s+thích|Liên\s+kết\s+ngoài|Xem\s+thêm|"
        r"Tài\s+liệu\s+tham\s+khảo|Thư\s+mục|Đọc\s+thêm"
        r")\s*=+\s*$",
        re.IGNORECASE | re.MULTILINE
    )

    match = terminal_regex.search(text)
    if match:
        # Nếu tìm thấy, chỉ lấy phần văn bản từ đầu cho đến ngay trước tiêu đề đó
        text = text[:match.start()]

    return text

def remove_wiki_images(text: str) -> str:
    """Xóa toàn bộ khối hình ảnh và mô tả ảnh có chứa dấu ngoặc lồng nhau."""

    # Các tiền tố đánh dấu hình ảnh trong Wiki
    image_prefixes = ["[[Tập tin:", "[[File:", "[[Hình:", "[[Image:"]

    for prefix in image_prefixes:
        while True:
            start_idx = text.find(prefix)
            if start_idx == -1:
                break # Không tìm thấy nữa thì thoát vòng lặp

            # Bắt đầu đếm ngoặc từ vị trí tìm thấy
            open_brackets = 2  # Vì prefix đã có sẵn 2 dấu '['
            end_idx = start_idx + len(prefix)

            while end_idx < len(text) and open_brackets > 0:
                if text[end_idx:end_idx+2] == "[[":
                    open_brackets += 2
                    end_idx += 2
                elif text[end_idx:end_idx+2] == "]]":
                    open_brackets -= 2
                    end_idx += 2
                else:
                    end_idx += 1

            # Xóa toàn bộ đoạn từ start_idx đến end_idx
            text = text[:start_idx] + text[end_idx:]

    return text

def _remove_balanced_braces(text: str) -> str:
    """Xóa toàn bộ nội dung trong {{...}} (templates). Xử lý template lồng nhau."""
    pattern = re.compile(r"\{\{[^{}]*\}\}")
    while True:
        new_text = pattern.sub("", text)
        if new_text == text: break
        text = new_text
    return text

def _remove_wiki_tables(text: str) -> str:
    """Xóa toàn bộ nội dung trong {|...|} (wiki tables)."""
    pattern = re.compile(r"\{\|[^\{\}]*\|\}")
    while True:
        new_text = pattern.sub("", text)
        if new_text == text: break
        text = new_text
    return text

def _remove_balanced_brackets(text: str) -> str:
    """Xử lý [[...]] links: giữ display text, xóa File/Image/Category."""
    text = re.compile(r"\[\[(?:File|Tập tin|Image|Hình|Category|Thể loại):.*?\]\]", re.IGNORECASE).sub("", text)
    text = re.compile(r"\[\[[^\]\|]+\|([^\]]+)\]\]").sub(r"\1", text)
    text = re.compile(r"\[\[([^\]]+)\]\]").sub(r"\1", text)
    return text

def _remove_single_brackets(text: str) -> str:
    """Xử lý [...] external links: giữ label, xóa URL."""
    text = re.sub(r"\[([a-z][a-z0-9_-]*):([^\]\|]*)\|([^\]]+)\]", r"\3", text)
    text = re.sub(r"\[[a-z]{2}:[^\]]+\]", "", text)
    text = re.sub(r"\[[a-z][a-z0-9_-]*:([^\]]+)\]", r"\1", text)
    text = re.sub(r"\[https?://[^\s\]]+\s+([^\]]+)\]", r"\1", text)
    text = re.sub(r"\[https?://[^\]]+\]", "", text)
    return text

def _strip_list_prefixes(text: str) -> str:
    """Xóa ký tự list prefix (*, #, ;, :) ở đầu dòng."""
    lines = text.splitlines(); out = []
    for line in lines:
        stripped = line.lstrip()
        m = _LIST_PREFIX.match(stripped)
        if m:
            prefix = m.group(1); content = stripped[m.end():]
            if ";" in prefix and ": " in content:
                term, _, defn = content.partition(": ")
                content = term.strip()
                if defn.strip(): content += ": " + defn.strip()
            content = re.sub(r"^[\s:]+", "", content)
            out.append(content)
        else: out.append(line)
    return "\n".join(out)

def _clean_dangling_punctuation(text: str) -> str:
    """Xử lý các dấu câu sau khi xóa template."""

    # 1. Xóa dấu [, ; :] đứng ngay sau dấu ngoặc mở
    text = re.sub(r'\(\s*[,;:]+\s*', '(', text)

    # 2. Xóa dấu [, ; :] đứng ngay trước dấu ngoặc đóngỹ)"
    text = re.sub(r'\s*[,;:]+\s*\)', ')', text)

    # 3. Xóa hoàn toàn các cặp ngoặc rỗng hoặc chỉ chứa khoảng trắng
    text = re.sub(r'\(\s*\)', '', text)

    # 4. Gộp các dấu câu lặp lại liên tiếp do xóa phần tử ở giữa
    text = re.sub(r'([,;])\s*[,;]+', r'\1', text)

    # 5. Xử lý khoảng trắng thừa trước dấu phẩy hoặc chấm
    text = re.sub(r'\s+([,.?!;:])', r'\1', text)

    return text

def clean_wikitext(raw: str) -> str:
    """Làm sạch hoàn chỉnh một chuỗi wikitext thành plain text."""
    text = raw
    text = _remove_balanced_braces(text)
    text = _remove_wiki_tables(text)
    text = _remove_balanced_brackets(text)
    text = _remove_single_brackets(text)

    for pattern, repl in _PATTERNS:
        text = pattern.sub(repl, text)

    text = _MARKUP.sub("", text)
    text = _MAGIC.sub("", text)
    text = _HEADERS.sub(r"\1", text)
    text = _EMPTY_HEADERS.sub("", text)
    text = _strip_list_prefixes(text)

    m = _TERMINAL_SECTION_RE.search(text)
    if m: text = text[:m.start()]

    text = _html.unescape(text)
    text = _clean_dangling_punctuation(text)
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

# --- Helpers Deduplicate ---
MIN_PARA_CHARS = 50
MIN_DOC_CHARS = 20

def normalize_text(text: str) -> str:
    """Chuẩn hóa Unicode NFC để hash nhất quán."""
    if text is None: return ""
    return unicodedata.normalize("NFC", text)

def sha_bytes(text):
    """Trả về SHA-256 digest (bytes) của text đã normalize."""
    normalized = normalize_text(text)
    return hashlib.sha256(normalized.encode('utf-8')).digest()

def dedup_paragraphs(text, seen_paras):
    """Loại bỏ các đoạn văn trùng lặp (>= MIN_PARA_CHARS) trong document."""
    paragraphs = text.split('\n')
    kept_paragraphs = []

    for p in paragraphs:
        p_strip = p.strip()
        if len(p_strip) < MIN_PARA_CHARS:
            kept_paragraphs.append(p)
            continue

        p_hash = sha_bytes(p_strip)
        if p_hash not in seen_paras:
            seen_paras.add(p_hash)
            kept_paragraphs.append(p)

    return '\n'.join(kept_paragraphs)

def flush_rows(writer, out_path, rows):
    """Ghi batch rows vào Parquet file."""
    if not rows: return writer
    table = pa.table({"text": rows})
    if writer is None:
        writer = pq.ParquetWriter(out_path, table.schema, compression="snappy")
    writer.write_table(table)
    rows.clear()
    return writer

def process_and_dedup(input_path: Path, output_dir: Path):
    """Đọc JSONL raw, làm sạch, dedup và ghi ra Parquet trực tiếp."""
    output_dir.mkdir(parents=True, exist_ok=True)
    out_path = output_dir / "vi_wiki_clean_dedup.parquet"

    if out_path.exists(): out_path.unlink()

    seen_docs_raw, seen_docs_final, para_seen = set(), set(), set()
    total_original, total_kept = 0, 0
    writer, out_rows = None, []
    BATCH_SIZE = 10_000

    with input_path.open("r", encoding="utf-8") as in_f:
        for line in in_f:
            line = line.strip()
            if not line: continue

            total_original += 1
            record = json.loads(line)
            raw_text = record.get("text", "")

            # 1. Clean Wikitext
            clean_text = clean_wikitext(raw_text)

            if not clean_text: continue

            # 2. Dedup Doc-level 1
            raw_hash = sha_bytes(clean_text)
            if raw_hash in seen_docs_raw: continue
            seen_docs_raw.add(raw_hash)

            # 3. Dedup Paragraph-level
            new_text = dedup_paragraphs(clean_text, para_seen)
            if len(new_text) < MIN_DOC_CHARS: continue

            # 4. Dedup Doc-level 2
            final_hash = sha_bytes(new_text)
            if final_hash in seen_docs_final: continue
            seen_docs_final.add(final_hash)

            # Lưu lại
            out_rows.append(new_text)
            total_kept += 1

            if len(out_rows) >= BATCH_SIZE:
                writer = flush_rows(writer, out_path, out_rows)
                logger.info(f"Đã xử lý {total_original} bài viết (giữ lại {total_kept})...")

    writer = flush_rows(writer, out_path, out_rows)
    if writer is not None: writer.close()
    logger.info(f"Hoàn tất. Total: {total_original} -> Kept: {total_kept}. Đã lưu tại {out_path}")

# --- Chạy pipeline Làm sạch & Dedup ---
INPUT_JSONL = Path("/content/drive/MyDrive/DM/data/raws/vi_wiki_articles.jsonl")
TRAIN_DIR = Path("/content/drive/MyDrive/DM/data/train")

if INPUT_JSONL.exists():
    process_and_dedup(INPUT_JSONL, TRAIN_DIR)
else:
    logger.error("Chưa thấy file raw. Vui lòng chạy cell crawl Wiki trước.")

In [8]:
import random as rd

clean_path = "/content/drive/MyDrive/DM/data/train/vi_wiki_clean_dedup.parquet"
df_clean = pd.read_parquet(clean_path)

print("\nTrích đoạn bài viết ngẫu nhiên:")
print(df_clean['text'].iloc[rd.randint(0, 49998)][:300] + "...")


Trích đoạn bài viết ngẫu nhiên:
Farnaz Shetty (sinh ngày 16 tháng 9 năm 1991 tại Mumbai, Ấn Độ) là một diễn viên truyền hình người Ấn Độ. Cô từng tốt nghiệp đại học Mithibai chuyên ngành hội họa và có ý định trở thành một họa sĩ nhưng sau đó chuyển sang diễn xuất. Năm 2013 cô đảm nhận vai diễn đầu tay trong sự nghiệp diễn xuất của...


## Phần 3: Tokenization, Xử lý <UNK>, Split Dataset và Lưu Artifacts

In [3]:
!pip install pyvi pandas scikit-learn tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 54.0 MB/s eta 0:00:00


In [5]:
import os
import re
import pickle
import pandas as pd
import numpy as np
from typing import List, Dict, Tuple, Set
from collections import Counter
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from pyvi import ViTokenizer
import logging

# Cấu hình logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

# --- CÁC ĐƯỜNG DẪN CỐ ĐỊNH TỪ PIPELINE TRƯỚC ---
CLEAN_PARQUET_PATH = "/content/drive/MyDrive/DM/data/train/vi_wiki_clean_dedup.parquet"
OUTPUT_DIR = "/content/drive/MyDrive/DM/data/train"

# ==================================================
# PHẦN 1 — START/END TOKEN & TOKENIZATION
# ==================================================
def simple_sent_tokenize(text: str) -> List[str]:
    """
    Tách câu đơn giản bằng Regex (dựa vào dấu chấm, chấm hỏi, chấm than).
    Dùng để thay thế sent_tokenize của underthesea nhằm tối ưu tốc độ.
    """
    # Tách dựa trên dấu kết thúc câu và khoảng trắng theo sau
    sentences = re.split(r'(?<=[.!?])\s+', str(text).strip())
    return [s.strip() for s in sentences if s.strip()]

def tokenize_and_pad(text: str, n_gram: int) -> List[List[str]]:
    """
    Tách câu, làm sạch, tách từ (Word-level bằng PyVi) và thêm padding tokens.
    Số lượng <START> phụ thuộc vào N (N-gram sẽ có N-1 token <START>).
    """
    sentences = simple_sent_tokenize(text)
    padded_sentences = []

    # Số lượng <START> token cần thiết cho cửa sổ N-gram
    n_start = max(1, n_gram - 1)

    for sentence in sentences:
        # Tiền xử lý: Giữ lại chữ cái, số và khoảng trắng, chuyển in thường
        clean_sentence = re.sub(r'[^\w\s]', ' ', sentence.lower())

        # Tách từ bằng PyVi (Từ ghép sẽ được nối bằng dấu '_', VD: 'học_sinh')
        tokenized_string = ViTokenizer.tokenize(clean_sentence)
        tokens = tokenized_string.split()

        if not tokens:
            continue

        # Padding <START> và <END>
        padded = ['<START>'] * n_start + tokens + ['<END>']
        padded_sentences.append(padded)

    return padded_sentences

# ==================================================
# PHẦN 2 — UNKNOWN TOKEN (<UNK>)
# ==================================================
def apply_unk_and_build_vocab(
    corpus: List[List[str]],
    threshold: int
) -> Tuple[List[List[str]], Dict[str, int], Set[str]]:
    """
    Thống kê tần suất từ vựng, chuyển các từ hiếm (freq < threshold) thành <UNK>.
    """
    logger.info("Đang đếm tần suất các từ trong toàn bộ corpus...")
    freq_counter = Counter()
    for sentence in tqdm(corpus, desc="Counting frequencies"):
        freq_counter.update(sentence)

    # Đảm bảo special tokens không bao giờ bị biến thành <UNK>
    special_tokens = {'<START>', '<END>', '<UNK>'}
    for st in special_tokens:
        if st in freq_counter:
            freq_counter[st] = float('inf')

    # Lọc vocabulary (chỉ giữ từ có tần suất >= threshold)
    vocab = {word for word, count in freq_counter.items() if count >= threshold}
    vocab.update(special_tokens)

    logger.info(f"Kích thước Vocabulary gốc: {len(freq_counter):,}")
    logger.info(f"Kích thước Vocabulary sau ngưỡng ({threshold}): {len(vocab):,}")

    # Áp dụng <UNK>
    logger.info("Đang thay thế từ hiếm bằng token <UNK>...")
    processed_corpus = []
    for sentence in tqdm(corpus, desc="Applying <UNK>"):
        new_sentence = [word if word in vocab else '<UNK>' for word in sentence]
        processed_corpus.append(new_sentence)

    # Tính lại dictionary frequency chuẩn xác sau khi gộp <UNK>
    final_freq = Counter()
    for sentence in processed_corpus:
        final_freq.update(sentence)

    return processed_corpus, dict(final_freq), vocab

# ==================================================
# PHẦN 3 — SPLIT DATASET
# ==================================================
def split_dataset(
    corpus: List[List[str]],
    test_ratio: float = 0.1,
    val_ratio: float = 0.1,
    random_state: int = 42
) -> Tuple[List[List[str]], List[List[str]], List[List[str]]]:
    """
    Chia dataset thành Train/Val/Test với tỷ lệ chuẩn (VD: 80/10/10).
    Sử dụng random seed để kết quả tái tạo được.
    """
    logger.info("Đang phân chia Train / Val / Test (80/10/10)...")

    # Tính tỷ lệ validation tương đối trên phần còn lại sau khi tách Test
    relative_val_size = val_ratio / (1.0 - test_ratio)

    train_val, test = train_test_split(
        corpus, test_size=test_ratio, random_state=random_state
    )
    train, val = train_test_split(
        train_val, test_size=relative_val_size, random_state=random_state
    )

    logger.info(f"Kích thước tập Train: {len(train):,} câu")
    logger.info(f"Kích thước tập Val:   {len(val):,} câu")
    logger.info(f"Kích thước tập Test:  {len(test):,} câu")

    return train, val, test

# ==================================================
# PHẦN 4 & 5 — OUTPUT FORMAT & SAVE FILE
# ==================================================
def save_artifacts(
    train: List[List[str]],
    val: List[List[str]],
    test: List[List[str]],
    vocab: Set[str],
    word_freq: Dict[str, int],
    output_dir: str
) -> None:
    """Lưu toàn bộ artifacts ra file pickle (.pkl)."""
    os.makedirs(output_dir, exist_ok=True)

    files_to_save = {
        "train.pkl": train,
        "val.pkl": val,
        "test.pkl": test,
        "vocab.pkl": vocab,
        "word_freq.pkl": word_freq
    }

    for filename, data in files_to_save.items():
        filepath = os.path.join(output_dir, filename)
        logger.info(f"Đang lưu {filename}...")
        with open(filepath, 'wb') as f:
            pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)

    logger.info(f"Hoàn tất lưu toàn bộ file tại: {output_dir}")

# ==================================================
# PHẦN 6 — PIPELINE EXECUTION (MAIN)
# ==================================================
def run_corpus_preparation_pipeline(
    parquet_path: str = CLEAN_PARQUET_PATH,
    output_dir: str = OUTPUT_DIR,
    n_gram: int = 3,           # Hỗ trợ N-gram tổng quát (VD: Trigram)
    unk_threshold: int = 2,    # Threshold loại bỏ từ hiếm
    max_docs: int = None       # Tối ưu RAM: Giới hạn số bài viết (nếu cần test)
) -> None:
    """
    Chạy toàn bộ quá trình xử lý Corpus từ file Parquet -> PKL.
    Tối ưu memory bằng cách giải phóng (del) các biến lớn sau khi sử dụng.
    """
    logger.info("=== BẮT ĐẦU PREPROCESSING CORPUS BẰNG PYVI ===")

    # 1. Đọc dữ liệu
    if not os.path.exists(parquet_path):
        logger.error(f"Không tìm thấy file {parquet_path}. Chạy pipeline Crawl trước!")
        return

    logger.info(f"Đọc dữ liệu từ {parquet_path}")
    df = pd.read_parquet(parquet_path)
    if max_docs:
        df = df.head(max_docs)

    # 2. Tokenize & Padding
    corpus_raw = []
    for text in tqdm(df['text'].dropna(), desc="Tokenization & Padding (PyVi)"):
        corpus_raw.extend(tokenize_and_pad(text, n_gram))

    del df # Tối ưu Memory

    # 3. Handle <UNK> & Build Vocab
    corpus_unk, word_freq, vocab = apply_unk_and_build_vocab(corpus_raw, unk_threshold)
    del corpus_raw # Tối ưu Memory

    # 4. Split Train/Val/Test
    train_data, val_data, test_data = split_dataset(corpus_unk)
    del corpus_unk # Tối ưu Memory

    # --- OUTPUT FORMAT THEO YÊU CẦU ---
    logger.info("\n" + "="*50)
    logger.info("DATASET STATISTICS & OUTPUT FORMAT")
    logger.info("="*50)
    logger.info(f"- Tổng số từ vựng (Vocab size): {len(vocab):,}")
    logger.info(f"- Cấu trúc Output (List of Lists): {type(train_data)} chứa {type(train_data[0])}")

    # Print mẫu dữ liệu
    sample_idx = 0
    while len(train_data[sample_idx]) < 5 and sample_idx < len(train_data):
        sample_idx += 1 # Tìm một câu có ý nghĩa để in mẫu

    logger.info(f"- Ví dụ 1 câu sau xử lý: {train_data[sample_idx]}")
    logger.info("="*50 + "\n")

    # 5. Save Artifacts
    save_artifacts(train_data, val_data, test_data, vocab, word_freq, output_dir)
    logger.info("=== PIPELINE HOÀN TẤT THÀNH CÔNG! ===")

# Thực thi Pipeline
if __name__ == "__main__":
    run_corpus_preparation_pipeline(
        n_gram=3,
        unk_threshold=2,
        max_docs=None
    )

Tokenization & Padding (PyVi):   0%|          | 0/49989 [00:00<?, ?it/s]

Counting frequencies:   0%|          | 0/1116637 [00:00<?, ?it/s]

Applying <UNK>:   0%|          | 0/1116637 [00:00<?, ?it/s]